# OpenPlaque — BACCE downstream seed + corrected tracker interface

Self-contained notebook cells; no `%run`.

This starts from the **ostium candidate produced by the BACCE compatibility run**, reconstructs a short image-supported proximal RCA route, moves the learned-tracker seed ~6 mm downstream, estimates direction from several route points, and uses the authors' intended **500 action channels + 1 radius channel** split.

The ostium remains the 0-mm landmark.


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/OpenPlaque /content/BACCE
!git clone -q --branch bacce-downstream-seed-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git clone -q https://github.com/514sz/Branch-aware-centerline-extraction.git /content/BACCE
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas torch


In [ ]:
import sys, shutil
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk, torch
from scipy import ndimage as ndi
from skimage.filters import frangi
from skimage.graph import route_through_array

sys.path.insert(0,'/content/OpenPlaque/src')
sys.path.insert(0,'/content/BACCE')
from openplaque.study import OpenPlaqueStudy
from Net import Tracker_Net, Detector_Net
from utils import create_actions

ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUT=ROOT/'BACCE_Downstream_Seed'; OUT.mkdir(parents=True,exist_ok=True)


## 1. Load series 7, cached aorta, and the current ostium candidate

In [ ]:
# Source CCTA
dz=ROOT/'Full_DICOM.zip'; lz=Path('/content/Full_DICOM.zip')
if not lz.exists() or lz.stat().st_size!=dz.stat().st_size: shutil.copyfile(dz,lz)
shutil.rmtree('/content/full_dicom_bacce_ds',ignore_errors=True)
study=OpenPlaqueStudy(str(lz),extract_root='/content/full_dicom_bacce_ds')
img,ct,_=study.load_series(7); ct=np.asarray(ct)
sp_xyz=np.array(img.GetSpacing(),float); sp_zyx=sp_xyz[::-1]

# TotalSegmentator aorta
ap=ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz'
if not ap.exists(): raise FileNotFoundError(ap)
ai=sitk.ReadImage(str(ap))
if ai.GetSize()!=img.GetSize() or not np.allclose(ai.GetSpacing(),img.GetSpacing()):
    ai=sitk.Resample(ai,img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
aorta=sitk.GetArrayFromImage(ai)>0
da_full=ndi.distance_transform_edt(~aorta,sampling=sp_zyx)

# Use the ostium hypothesis from the compatibility notebook the user just ran.
prev=ROOT/'BACCE_Compatibility'/'bacce_compatibility_summary.csv'
if not prev.exists():
    raise FileNotFoundError('Run RCA_BACCE_Compatibility_SelfContained.ipynb once first: '+str(prev))
pr=pd.read_csv(prev).iloc[0]
ostium=np.array([pr.seed_z,pr.seed_y,pr.seed_x],float)
print('ostium hypothesis z,y,x:',ostium)
print('previous score/length/radius:',
      float(pr.candidate_score),float(pr.candidate_length_mm),float(pr.candidate_radius_mm))


## 2. Reconstruct an image-supported proximal route

In [ ]:
o=np.round(ostium).astype(int)
half=np.ceil(np.array([16.,34.,34.])/sp_zyx).astype(int)
lo=np.maximum(0,o-half); hi=np.minimum(np.array(ct.shape),o+half+1)
roi=ct[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]].astype(np.float32)
ar=aorta[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]
sl=ostium-lo
da=ndi.distance_transform_edt(~ar,sampling=sp_zyx)

# Local intensity + vesselness support
sm=ndi.gaussian_filter(roi,sigma=np.maximum(.65/sp_zyx,.5))
a_hu=ct[aorta & (np.indices(ct.shape)[0]>=225) & (np.indices(ct.shape)[0]<=305)]
med=float(np.median(a_hu))
blood_thr=float(np.clip(.43*med,180,360))
lo_hu=max(140.,.55*blood_thr); hi_hu=max(lo_hu+100.,1.35*med)
ints=np.clip((sm-lo_hu)/(hi_hu-lo_hu),0,1)
v=np.nan_to_num(frangi(sm,sigmas=(1,2,3),black_ridges=False))
v99=np.percentile(v[v>0],99) if np.any(v>0) else 1.
vn=np.clip(v/max(v99,1e-6),0,1)
support=.58*ints+.42*vn
cost=1/(.06+support); cost[ar]+=40; cost[da<.6]+=15; cost[sm<lo_hu]+=12
cost=cost.astype(np.float32)

# Initial outward direction: from local aortic center toward ostium.
z=int(round(ostium[0])); yy,xx=np.where(aorta[z])
a_ctr=np.array([z,float(np.mean(yy)),float(np.mean(xx))])
axis_mm=(ostium-a_ctr)*sp_zyx
axis_mm/=np.linalg.norm(axis_mm)

# Distal candidates: 12-28 mm away, outward and no longer hugging aorta.
g=np.argwhere(np.ones_like(roi,bool))
dmm=(g-sl)*sp_zyx; rad=np.linalg.norm(dmm,axis=1)
dot=dmm@axis_mm
su=support[tuple(g.T)]; dd=da[tuple(g.T)]
m=(rad>=12)&(rad<=28)&(dot>=7)&(dd>=4)&(su>=np.percentile(support,82))
eg=g[m]
if len(eg)==0: raise RuntimeError('No distal route endpoints found')
es=support[tuple(eg.T)]+.02*da[tuple(eg.T)]
eps=[]
for j in np.argsort(es)[::-1]:
    p=eg[j]
    if all(np.linalg.norm((p-q)*sp_zyx)>=3 for q in eps): eps.append(p)
    if len(eps)>=24: break

def met(path):
    p=np.asarray(path,int); hu=roi[tuple(p.T)]; su=support[tuple(p.T)]; dd=da[tuple(p.T)]
    st=np.diff(p.astype(float),axis=0)*sp_zyx; seg=np.linalg.norm(st,axis=1); L=float(seg.sum())
    if len(st)>=2:
        u=st/np.maximum(np.linalg.norm(st,axis=1,keepdims=True),1e-6)
        turn=float(np.mean(np.arccos(np.clip(np.sum(u[:-1]*u[1:],axis=1),-1,1))))
    else: turn=np.pi
    outward=float(np.mean(np.diff(dd)>=-.35)) if len(dd)>1 else 0
    score=1.8*np.mean(su)+.45*outward+.018*min(L,25)+.025*min(float(dd[-1]),14)-.25*turn
    return dict(length_mm=L,mean_hu=float(np.mean(hu)),p10_hu=float(np.percentile(hu,10)),
                mean_support=float(np.mean(su)),end_dist_aorta_mm=float(dd[-1]),
                outward_fraction=outward,mean_turn_rad=turn,score=float(score))

routes=[]
start=tuple(np.round(sl).astype(int))
for ep in eps:
    try: p,tc=route_through_array(cost,start,tuple(ep),fully_connected=True,geometric=True)
    except Exception: continue
    mm=met(p)
    if 8<=mm['length_mm']<=35: routes.append({'path':np.asarray(p,int),**mm})
if not routes: raise RuntimeError('No usable proximal route')
best=max(routes,key=lambda r:r['score'])
path=best['path']+lo
display(pd.DataFrame([{k:v for k,v in r.items() if k!='path'} for r in sorted(routes,key=lambda r:r['score'],reverse=True)[:6]]))
print('best route:',{k:round(v,3) for k,v in best.items() if k!='path'})


## 3. Move seed ~6 mm downstream and estimate direction from several path points

In [ ]:
st=np.diff(path.astype(float),axis=0)*sp_zyx
s=np.r_[0,np.cumsum(np.linalg.norm(st,axis=1))]
target=6.0; i=int(np.argmin(abs(s-target))); seed=path[i].astype(float)

# Fit principal direction over roughly 3-9 mm of route.
w=np.where((s>=3)&(s<=9))[0]
if len(w)<3: w=np.arange(max(0,i-3),min(len(path),i+4))
pm=path[w].astype(float)*sp_zyx
_,V=np.linalg.eigh(np.cov(pm,rowvar=False)); fwd=V[:,-1]
ref=(path[min(len(path)-1,i+2)]-path[max(0,i-2)])*sp_zyx
if np.dot(fwd,ref)<0: fwd=-fwd
fwd/=np.linalg.norm(fwd)

seed2=(seed*sp_zyx+2*fwd)/sp_zyx
start_direction=seed2-seed

def patch19(arr,p):
    z,y,x=np.round(p).astype(int)
    q=arr[z-9:z+10,y-9:y+10,x-9:x+10]
    if q.shape!=(19,19,19): raise ValueError('19^3 patch clipped')
    return q.astype(np.float32)
patch=patch19(ct,seed); pt=torch.from_numpy(patch)[None,None]

print('ostium:',np.round(ostium,2))
print('downstream arc mm:',round(float(s[i]),2))
print('seed1:',np.round(seed,2))
print('seed2:',np.round(seed2,2))
print('direction z,y,x:',np.round(start_direction,3))
print('patch:',pt.shape)


## 4. Correct BACCE tracker output split and optional checkpoint inference

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tracker=Tracker_Net(n_actions=500).to(device).eval()
detector=Detector_Net().to(device).eval()

with torch.no_grad():
    qr=tracker(pt.to(device)); dr=detector(pt.to(device))
assert tuple(qr.shape)==(1,501,1,1,1)
assert tuple(dr.shape)==(1,3,1,1,1)
print('architecture PASS:',tuple(qr.shape),tuple(dr.shape))
print('Interpretation: q[0:500]=directional Q values; q[500]=tracker radius')

ck=ROOT/'BACCE_Checkpoints'; tp=ck/'Tracker.pth'; dp=ck/'Detector.pth'
have=tp.exists() and dp.exists(); inf={}
if have:
    tracker.load_state_dict(torch.load(tp,map_location=device))
    detector.load_state_dict(torch.load(dp,map_location=device))
    tracker.eval(); detector.eval()
    with torch.no_grad():
        q=tracker(pt.to(device)).flatten(); d=detector(pt.to(device)).flatten()
    aq=q[:500]; tr=float(q[500].cpu()); prob=torch.softmax(aq,dim=0)
    topv,topi=torch.topk(prob,5)
    actions=create_actions(500,dis=1.5,spacing=sp_xyz).astype(float)
    rows=[]
    for idx,pv in zip(topi.cpu().tolist(),topv.cpu().tolist()):
        av=actions[idx]; avm=av*sp_zyx
        cos=float(np.dot(avm/np.linalg.norm(avm),fwd)) if np.linalg.norm(avm)>0 else -1
        rows.append((idx,float(pv),cos,av.tolist()))
    forward=[r for r in rows if r[2]>0]
    chosen=max(forward,key=lambda r:r[1]) if forward else max(rows,key=lambda r:r[2])
    inf=dict(chosen_action=chosen[0],chosen_prob=chosen[1],chosen_forward_cos=chosen[2],
             tracker_radius=tr,bifurcation_prob=float(torch.sigmoid(d[0]).cpu()),
             endpoint_prob=float(torch.sigmoid(d[1]).cpu()),detector_radius=float(d[2].cpu()))
    print('CHECKPOINTS LOADED'); print('top actions (idx,prob,forward cosine,offset):')
    for r in rows: print(r)
    print('chosen:',chosen); print(inf)
else:
    print('No checkpoints yet; interface is ready.')


## 5. Visual report and saved initialization route

In [ ]:
zs=int(round(seed[0])); y=int(round(seed[1])); x=int(round(seed[2]))
fig,ax=plt.subplots(1,4,figsize=(20,5))
ax[0].imshow(ct[zs],cmap='gray',vmin=-200,vmax=900)
ax[0].contour(aorta[zs].astype(float),levels=[.5],linewidths=1)
sel=abs(path[:,0]-zs)<=2
ax[0].plot(path[sel,2],path[sel,1],'.-'); ax[0].plot(seed[2],seed[1],'s')
ax[0].set_title(f'Route + 6 mm BACCE seed z={zs}'); ax[0].axis('off')

r=55; y0=max(0,y-r);y1=min(ct.shape[1],y+r);x0=max(0,x-r);x1=min(ct.shape[2],x+r)
ax[1].imshow(ct[zs,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=900)
ax[1].plot(path[sel,2]-x0,path[sel,1]-y0,'.-'); ax[1].plot(seed[2]-x0,seed[1]-y0,'s')
ax[1].set_title('Downstream seed close-up'); ax[1].axis('off')

ax[2].imshow(patch[9],cmap='gray',vmin=-200,vmax=900); ax[2].plot(9,9,'s')
ax[2].set_title('BACCE 19×19×19 center'); ax[2].axis('off')

dd=da_full[tuple(path.astype(int).T)]
ax[3].plot(s,dd,label='distance to aorta')
ax[3].axvline(float(s[i]),ls='--',label='BACCE seed')
ax[3].set_xlabel('arc length from ostium (mm)'); ax[3].legend(); ax[3].set_title('Route QC')

plt.tight_layout()
rp=OUT/'bacce_downstream_seed_report.png'; fig.savefig(rp,dpi=180,bbox_inches='tight'); plt.show()

summary={'route_length_mm':best['length_mm'],'route_mean_hu':best['mean_hu'],
         'route_p10_hu':best['p10_hu'],'route_mean_support':best['mean_support'],
         'route_end_dist_aorta_mm':best['end_dist_aorta_mm'],
         'route_outward_fraction':best['outward_fraction'],
         'seed_arc_mm':float(s[i]),'seed_z':seed[0],'seed_y':seed[1],'seed_x':seed[2],
         'seed2_z':seed2[0],'seed2_y':seed2[1],'seed2_x':seed2[2],
         'checkpoints_present':bool(have),**inf}
pd.DataFrame([summary]).to_csv(OUT/'bacce_downstream_seed_summary.csv',index=False)
np.savetxt(OUT/'proximal_initialization_route_zyx.txt',path,fmt='%d')
display(pd.DataFrame([summary]))
print('Saved',rp)
